# A.3 — LoRA: từ cập nhật hạng thấp đến lớp tuyến tính suy luận
Chạy tuần tự trên CPU với PyTorch và bản repo chứa `src/lora.py`, `tests/test_lora.py`. Không tải mô hình, cài PEFT hay dùng dữ liệu thủ tục. [Hu et al., §4.1, Eq. (3)](https://arxiv.org/abs/2106.09685); [chứng minh B.4](../docs/theory/part-b.pdf).

**Dẫn xuất.** Cho lớp đã có trọng số $W_0\in\mathbb R^{d\times k}$, bias $b$. Thay vì tối ưu lại mọi phần tử của $W_0$, xét $\min_{\operatorname{rank}(\Delta W)\le r}\sum_i\ell((W_0+\Delta W)x_i+b,t_i)$ với $W_0,b$ cố định. Ma trận có hạng không quá $r$ viết được dưới dạng $BA$ (chẳng hạn từ SVD thu gọn, đệm cột/hàng 0 nếu cần). Đặt $\Delta W=(\alpha/r)BA$, với $A\in\mathbb R^{r\times k}$, $B\in\mathbb R^{d\times r}$, $\alpha>0$. Hệ số khác 0 không đổi tập ma trận biểu diễn được. Chỉ có $rk+dr$ phần tử học, ít hơn $dk$ khi $r(d+k)<dk$.

`LoRALinear.forward` tính $W_0x+b+(\alpha/r)B(Ax)$, không dựng $BA$ khi học. Khởi tạo $A$ ngẫu nhiên và $B=0$ giữ nguyên hàm ban đầu. Lúc này gradient của $A$ bằng 0 vì đi qua $B$, nhưng $B$ có thể nhận gradient ngay ở bước đầu. Đóng băng lớp nền không có nghĩa chặn gradient theo đầu vào.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys
import torch
from torch import nn

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/lora.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Hãy chuyển thư mục làm việc đến repo chứa src/lora.py và tests/.')
sys.path.insert(0, str(root))
from src.lora import LoRALinear

torch.manual_seed(42)
k, d, rank, alpha = 12, 8, 2, 6.
base = nn.Linear(k, d, dtype=torch.float64)
x = torch.randn(6, k, dtype=base.weight.dtype)
target = torch.randn(6, d, dtype=x.dtype)
base_out = base(x).detach()
layer = LoRALinear(base, rank, alpha)
count = sum(p.numel() for p in layer.parameters() if p.requires_grad)
assert count == rank * (d + k)
torch.testing.assert_close(base_out, layer(x), atol=0, rtol=0)
print(f'Python {platform.python_version()}, torch {torch.__version__}, CPU float64, seed=42')
print(f'{count} tham số học; {k*d} trọng số nền và {d} phần tử bias đóng băng.')

**Tạo cập nhật thật cho phép thử.** Dùng lỗi bình phương trung bình trên tensor minh họa (tương ứng negative log-likelihood Gaussian với phương sai cố định, sai khác một hệ số và hằng số). Hai bước SGD giúp quan sát cả $B$ và $A$ nhận gradient. Đây không phải thí nghiệm tinh chỉnh mô hình ngôn ngữ.

In [ ]:
for step in range(2):
    layer.zero_grad(set_to_none=True)
    (layer(x) - target).square().mean().backward()
    with torch.no_grad():
        for factor in (layer.A, layer.B):
            factor.add_(factor.grad, alpha=-0.05)

with torch.no_grad():
    before = layer(x)
    delta = layer.scale * (layer.B @ layer.A)
    observed_rank = torch.linalg.matrix_rank(delta).item()
    assert 0 < observed_rank <= rank and not torch.allclose(before, base_out)
    print(f'Hạng cập nhật: {observed_rank}; sai khác lớn nhất so với lớp nền: {(before - base_out).abs().max().item():.6g}')

**Merge để suy luận.** Tính $W'=W_0+(\alpha/r)BA$ một lần. `merged = layer.merge()` trả một `nn.Linear` riêng, giữ nguyên bias và không còn nhánh adapter. Dùng **`merged(x)`** để suy luận; `layer` gốc vẫn dùng được để học tiếp. Hai cách nhân tương đương theo đại số, có thể lệch nhẹ do thứ tự phép toán dấu phẩy động. Không suy ra số đo tốc độ hoặc VRAM từ ví dụ này.

In [ ]:
merged = layer.merge()
with torch.no_grad():
    after = merged(x)
    torch.testing.assert_close(after, before, rtol=1e-10, atol=1e-12)
    print(f'Sai khác lớn nhất trước/sau merge: {(after - before).abs().max().item():.3g}')

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_lora.py', '-v'], cwd=root, check=True);